## PHASE 1 — SETUP

In [ ]:
# Install all required packages with pinned versions
#
# WHY PINNED VERSIONS?
# transformers, peft, and trl release frequently and often break each other.
# These exact versions are confirmed compatible with Phi-4 Mini on Kaggle.
#
# ⚠️  AFTER THIS CELL COMPLETES:
#     Go to Run → Restart & clear cell outputs
#     Then continue from Cell 2 — do NOT re-run this cell

!pip install -q \
    transformers==4.49.0 \
    trl==0.16.0 \
    peft==0.14.0 \
    accelerate==1.3.0 \
    bitsandbytes \
    datasets \
    huggingface_hub \
    einops \
    rouge-score

print("All packages installed")
print("Now go to Run → Restart & clear cell outputs, then continue from Cell 2")

In [1]:
# Login to Hugging Face
#
# WHY: Phi-4 Mini requires authentication to download.
# We read the token from Kaggle Secrets (Add-ons → Secrets → HF_TOKEN)
# instead of hardcoding it — never paste tokens directly in notebooks!

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets   = UserSecretsClient()
hf_token  = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

print("Logged in to Hugging Face")

Logged in to Hugging Face


In [2]:
# Import all libraries
#
# WHY THE sys.modules CLEANUP:
# Kaggle pre-installs its own version of transformers in the base image.
# Without clearing the module cache, Python loads the wrong (older) version
# even after we've installed the correct one — causing 'is_flax_available'
# ImportErrors. This fix forces a clean reload from the right location.

import sys

# Clear any stale cached imports from Kaggle's base environment
mods_to_remove = [
    k for k in sys.modules
    if any(x in k for x in ['transformers', 'peft', 'trl'])
]
for mod in mods_to_remove:
    del sys.modules[mod]

# Ensure our installed versions take priority in the import path
sys.path.insert(0, '/usr/local/lib/python3.12/dist-packages')

# Now import everything cleanly
import torch
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

# Prevents GPU memory fragmentation during training
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("All libraries imported successfully")
print(f"   transformers : {__import__('transformers').__version__}")
print(f"   peft         : {__import__('peft').__version__}")
print(f"   trl          : {__import__('trl').__version__}")

2026-03-03 02:46:04.814981: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772505965.046842     125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772505965.109428     125 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772505965.653086     125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772505965.653129     125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772505965.653132     125 computation_placer.cc:177] computation placer alr

All libraries imported successfully
   transformers : 4.49.0
   peft         : 0.14.0
   trl          : 0.16.0


In [3]:
# Confirm GPU is available
#
# WHY: Without GPU, training would take days instead of hours.
# We need CUDA=True and 2x T4 GPUs before proceeding.
# If you see CUDA=False, go to Settings → Accelerator → GPU T4 x2

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
print()

for i in range(torch.cuda.device_count()):
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    free  = total - torch.cuda.memory_allocated(i) / 1e9
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | Total: {total:.1f}GB | Free: {free:.1f}GB")

# Stop here if GPU is not available
assert torch.cuda.is_available(), "No GPU detected! Enable GPU T4 x2 in Settings."

CUDA available : True
GPU count      : 2

  GPU 0: Tesla T4 | Total: 15.6GB | Free: 15.6GB
  GPU 1: Tesla T4 | Total: 15.6GB | Free: 15.6GB


## PHASE 2 — DATA PREPARATION

In [4]:
# Load the financial Q&A dataset
#
# DATASET: virattt/financial-qa-10K
# ~7,000 Q&A pairs extracted from real SEC 10-K annual filings.
# Each example has: question, answer, context (passage from the filing).
# This is ideal for teaching a model to reason about earnings reports,
# risk disclosures, balance sheets, and business performance.

dataset = load_dataset("virattt/financial-qa-10K")

print(dataset)
print("\n--- Raw Sample ---")
print(dataset["train"][0])

README.md:   0%|          | 0.00/419 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'context', 'ticker', 'filing'],
        num_rows: 7000
    })
})

--- Raw Sample ---
{'question': 'What area did NVIDIA initially focus on before expanding to other computationally intensive fields?', 'answer': 'NVIDIA initially focused on PC graphics.', 'context': 'Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.', 'ticker': 'NVDA', 'filing': '2023_10K'}


In [5]:
# Format data into Phi-4 Mini's chat template
#
# WHY FORMAT MATTERS:
# Phi-4 Mini was trained to expect a specific structure with special tokens:
#   <|system|>  — sets the assistant's role and persona
#   <|user|>    — the question or instruction
#   <|assistant|> — the answer the model should learn to produce
#   <|end|>     — marks the end of each section
#
# Getting this format wrong is one of the most common reasons
# fine-tuning produces poor results — the model never learns the task.
#
# We include the financial context (source passage from 10-K) in the
# user message so the model learns to ground answers in real documents.

def format_prompt(example):
    context  = example.get("context", "").strip()
    question = example["question"].strip()
    answer   = example["answer"].strip()

    # Include context if available (most examples have it)
    user_content = f"{question}\n\nContext: {context}" if context else question

    return {
        "text": f"""<|system|>
You are an expert financial analyst. Answer questions about company financials, earnings reports, and business performance accurately and concisely based on the provided context.<|end|>
<|user|>
{user_content}<|end|>
<|assistant|>
{answer}<|end|>"""
    }

formatted = dataset["train"].map(format_prompt)

print("Dataset formatted")
print("\n--- Formatted Sample (what the model will train on) ---")
print(formatted[0]["text"])

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dataset formatted

--- Formatted Sample (what the model will train on) ---
<|system|>
You are an expert financial analyst. Answer questions about company financials, earnings reports, and business performance accurately and concisely based on the provided context.<|end|>
<|user|>
What area did NVIDIA initially focus on before expanding to other computationally intensive fields?

Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.<|end|>
<|assistant|>
NVIDIA initially focused on PC graphics.<|end|>


In [6]:
# Split into train and test sets
#
# WHY: We hold out 10% of the data as a test set that the model
# NEVER sees during training. This makes evaluation meaningful —
# if we tested on training data, any improvement would be fake.
# seed=42 makes the split reproducible (same split every run).

split      = formatted.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
test_data  = split["test"]

print(f"Split complete")
print(f"   Train samples : {len(train_data)}")
print(f"   Test samples  : {len(test_data)}")

Split complete
   Train samples : 6300
   Test samples  : 700


## PHASE 3 — MODEL SETUP

In [7]:
# Configure 4-bit quantization (QLoRA)
#
# WHY 4-BIT QUANTIZATION:
# Phi-4 Mini in full 16-bit precision needs ~8GB of VRAM just to load.
# By compressing weights to 4-bit, we bring that down to ~2.5GB —
# leaving plenty of room for training on a 15GB T4 GPU.
#
# nf4 (NormalFloat4) is the best quantization format for LLMs.
# bfloat16 for compute keeps precision high during forward/backward passes.
# double_quant applies a second quantization to the quantization constants
# themselves, saving an extra ~0.3 bits per parameter.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit quantization config ready")

4-bit quantization config ready


In [8]:
# Load the tokenizer
#
# WHY: The tokenizer converts text → token IDs (numbers the model understands)
# and back. Phi-4 Mini has a 200K vocabulary — much larger than most LLMs,
# which makes it more efficient at multilingual and technical text.
#
# We set pad_token = eos_token because Phi-4 has no dedicated padding token.
# padding_side="right" is standard for causal (left-to-right) language models.

model_id = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded!")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Tokenizer loaded!
   Vocab size: 200,019


In [9]:
# Load Phi-4 Mini in 4-bit
#
# WHY device_map="auto":
# Automatically distributes model layers across both T4 GPUs intelligently.
# With 2x 15GB GPUs and a ~2.5GB quantized model, we have tons of headroom.
#
# prepare_model_for_kbit_training() does two things:
#   1. Enables gradient checkpointing (recomputes activations instead of
#      storing them all — trades speed for memory savings)
#   2. Casts layer norms to float32 for training stability

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

print("Phi-4 Mini loaded in 4-bit")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f"   GPU {i} memory used: {used:.2f} GB")

config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Phi-4 Mini loaded in 4-bit
   GPU 0 memory used: 3.08 GB
   GPU 1 memory used: 1.04 GB


In [10]:
# Attach LoRA adapters
#
# WHAT IS LoRA:
# Instead of updating all 3.8 billion parameters (expensive),
# LoRA injects small trainable matrices into specific layers.
# We train only ~1% of parameters — massively reducing memory and time.
#
# KEY HYPERPARAMETERS:
# r=16       — rank of the adapter matrices. Higher = more capacity but
#              more memory. 16 is a good balance for this task size.
# lora_alpha — scaling factor, convention is 2x the rank (so 32).
# target_modules — which layers to inject adapters into.
#              q/k/v/o are the attention layers (how the model "focuses").
#              gate/up/down are the feed-forward layers (how it "thinks").
# lora_dropout — small regularization to prevent overfitting.

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,912,896 || all params: 3,844,934,656 || trainable%: 0.2318


## PHASE 4 — TRAINING

In [ ]:
# Configure and start training
#
# KEY TRAINING SETTINGS EXPLAINED:
#
# num_train_epochs=3
#   The model sees the entire dataset 3 times.
#
# per_device_train_batch_size=4 + gradient_accumulation_steps=4
#   Effective batch size = 4 x 4 = 16. We can't fit 16 examples in
#   memory at once, so we process 4 at a time and accumulate gradients.
#
# learning_rate=2e-4
#   How fast the model updates weights. Too high = unstable training,
#   too low = slow learning. 2e-4 is standard for LoRA fine-tuning.
#
# bf16=True
#   Uses bfloat16 precision for speed. Must match the model's dtype.
#
# packing=True
#   Packs multiple short examples into one sequence so the GPU
#   processes more data per step — significant speed boost.
#
# optim="paged_adamw_8bit"
#   Memory-efficient optimizer that pages gradients to CPU when needed.
#
# gradient_checkpointing=True
#   Recomputes activations during backprop instead of storing them all.
#   Trades ~20% speed for significant memory savings.
#
# save_steps=200 + save_total_limit=2
#   Saves a checkpoint every 200 steps, keeping only the last 2.
#   Insurance against Kaggle session timeouts.

sft_config = SFTConfig(
    output_dir="./phi4-finance",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    report_to="none",
    max_seq_length=512,
    dataset_text_field="text",
    packing=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=test_data,
    args=sft_config,
)

print("Starting training... estimated 2–3 hours on 2x T4")
print("   Watch for loss decreasing from ~1.8 toward ~1.0 — that means it's learning!")
trainer.train()
print("Training complete")

In [12]:
import os

checkpoint_dir = "./phi4-finance"
if os.path.exists(checkpoint_dir):
    checkpoints = os.listdir(checkpoint_dir)
    print(f"✅ Checkpoints found: {checkpoints}")
else:
    print("❌ No checkpoints found — will need to restart training")

✅ Checkpoints found: ['checkpoint-200', 'checkpoint-294']


In [13]:
print("🔄 Resuming training from last checkpoint...")
trainer.train(resume_from_checkpoint=True)
print("✅ Training complete!")

🔄 Resuming training from last checkpoint...


Step,Training Loss,Validation Loss


✅ Training complete!


## PHASE 5 — SAVE & PUSH TO HUB

In [14]:
# Save the fine-tuned model locally
#
# WHY: We save only the LoRA adapter weights (~50–100MB),
# NOT the full 3.8B base model (which stays frozen and unchanged).
# This is what makes QLoRA practical — tiny adapter files, not gigabytes.

trainer.model.save_pretrained("./phi4-finance-final")
tokenizer.save_pretrained("./phi4-finance-final")

print("Model and tokenizer saved to ./phi4-finance-final")

Model and tokenizer saved to ./phi4-finance-final


In [15]:
# Push to Hugging Face Hub
#
# WHY: Publishing your adapter weights to HF Hub means:
#   - Anyone can load the model with 2 lines of code
#   - get a public URL to put on your resume/LinkedIn
#   - It's the industry standard for sharing fine-tuned models

trainer.model.push_to_hub("Emar7/phi4-finance-finetuned")
tokenizer.push_to_hub("Emar7/phi4-finance-finetuned")

print("Pushed to Hugging Face Hub!")
print("https://huggingface.co/Emar7/phi4-finance-finetuned")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to Hugging Face Hub!
https://huggingface.co/Emar7/phi4-finance-finetuned


## PHASE 6 — EVALUATION

We compare the **base Phi-4 Mini** vs our **fine-tuned version** on held-out test data.
Two types of evaluation:
1. **Qualitative** — side-by-side answer comparison (most compelling for portfolios)
2. **Quantitative** — ROUGE scores measuring answer overlap with reference answers

In [16]:
# Load both models for comparison
#
# We load two separate model instances:
#   eval_ft   — the fine-tuned model (base + our LoRA adapters)
#   base_only — vanilla Phi-4 Mini with no fine-tuning
# This lets us run the same question through both and compare answers.

from peft import PeftModel

# Fine-tuned model: load base then stack our LoRA adapter on top
eval_base = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
eval_ft = PeftModel.from_pretrained(eval_base, "./phi4-finance-final")
eval_ft.eval()

# Base model: plain Phi-4 Mini, no fine-tuning
base_only = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_only.eval()

print("Both models loaded for comparison!")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Both models loaded for comparison!


In [17]:
# Define the inference function
#
# This function takes any model + question and returns a generated answer.
# We use the same Phi-4 chat template as during training — consistency matters.
#
# temperature=0.1 — very low randomness, keeps answers focused and factual
# max_new_tokens=256 — limits response length (most financial answers are concise)
# We decode only the NEW tokens (not the prompt) by slicing from input length.

def generate_answer(model, tokenizer, question, context="", max_new_tokens=256):
    user_content = f"{question}\n\nContext: {context}" if context else question

    prompt = f"""<|system|>
You are an expert financial analyst. Answer questions about company financials, earnings reports, and business performance accurately and concisely based on the provided context.<|end|>
<|user|>
{user_content}<|end|>
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Slice off the prompt tokens — we only want the generated answer
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

print("Inference function ready")

Inference function ready


In [18]:
# Qualitative comparison: base vs fine-tuned
#
# These are general financial questions (no context needed).
# The fine-tuned model should give more structured, analyst-style answers
# using proper financial terminology — vs the base model being more generic.
# This side-by-side comparison is the most compelling part of your portfolio!

test_questions = [
    "What does a high debt-to-equity ratio indicate about a company?",
    "What is EBITDA and why do analysts use it?",
    "How do you interpret a company's free cash flow trend?",
    "What risks should investors watch for in a company's 10-K filing?",
    "What does it mean when a company's revenue grows but net income declines?",
]

print("=" * 70)
for i, question in enumerate(test_questions):
    print(f"\nQUESTION {i+1}:\n{question}")
    print("-" * 70)

    base_ans = generate_answer(base_only, tokenizer, question)
    print(f"BASE MODEL:\n{base_ans}")
    print()

    ft_ans = generate_answer(eval_ft, tokenizer, question)
    print(f"FINE-TUNED MODEL:\n{ft_ans}")
    print("=" * 70)


QUESTION 1:
What does a high debt-to-equity ratio indicate about a company?
----------------------------------------------------------------------
BASE MODEL:
A high debt-to-equity ratio indicates that a company has been aggressive in financing its growth with debt. This can result in volatile earnings due to the additional volatility that comes with financing growth with debt. It also indicates that the company is a riskier investment, as the company must cover interest payments on its debt. However, a high debt-to-equity ratio can also indicate that the company is confident in its ability to generate revenue and cash flow to cover its debt obligations.

FINE-TUNED MODEL:
A high debt-to-equity ratio indicates that a company has a high proportion of debt relative to its equity, which may suggest higher financial risk.

QUESTION 2:
What is EBITDA and why do analysts use it?
----------------------------------------------------------------------
BASE MODEL:
EBITDA stands for Earnings Bef

In [19]:
# ROUGE score evaluation on 100 test samples
#
# WHAT IS ROUGE:
# ROUGE measures word overlap between generated answers and reference answers.
#   ROUGE-1 = unigram (single word) overlap
#   ROUGE-2 = bigram (two word) overlap
#   ROUGE-L = longest common subsequence
#
# Higher scores = generated answers match the reference answers more closely.
# The improvement % is what goes in your README and blog post.
# This cell takes ~15–20 minutes (running 200 inferences total).

from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

eval_samples = test_data.select(range(100))
base_scores  = {"rouge1": [], "rouge2": [], "rougeL": []}
ft_scores    = {"rouge1": [], "rouge2": [], "rougeL": []}

print("Evaluating 100 test samples (~15–20 mins)...\n")

for i, sample in enumerate(eval_samples):
    question  = sample["question"]
    context   = sample.get("context", "")
    reference = sample["answer"]

    base_pred = generate_answer(base_only, tokenizer, question, context)
    ft_pred   = generate_answer(eval_ft,   tokenizer, question, context)

    base_s = scorer.score(reference, base_pred)
    ft_s   = scorer.score(reference, ft_pred)

    for metric in ["rouge1", "rouge2", "rougeL"]:
        base_scores[metric].append(base_s[metric].fmeasure)
        ft_scores[metric].append(ft_s[metric].fmeasure)

    if (i + 1) % 10 == 0:
        print(f"  ✅ {i+1}/100 evaluated...")

print("\nFINAL EVALUATION RESULTS")
print("=" * 58)
print(f"{'Metric':<12} {'Base Model':>12} {'Fine-Tuned':>12} {'Improvement':>12}")
print("-" * 58)
for metric in ["rouge1", "rouge2", "rougeL"]:
    base_avg = np.mean(base_scores[metric])
    ft_avg   = np.mean(ft_scores[metric])
    delta    = ((ft_avg - base_avg) / base_avg) * 100
    print(f"{metric:<12} {base_avg:>12.4f} {ft_avg:>12.4f} {delta:>11.1f}%")
print("=" * 58)

Evaluating 100 test samples (~15–20 mins)...

  ✅ 10/100 evaluated...
  ✅ 20/100 evaluated...
  ✅ 30/100 evaluated...
  ✅ 40/100 evaluated...
  ✅ 50/100 evaluated...
  ✅ 60/100 evaluated...
  ✅ 70/100 evaluated...
  ✅ 80/100 evaluated...
  ✅ 90/100 evaluated...
  ✅ 100/100 evaluated...

FINAL EVALUATION RESULTS
Metric         Base Model   Fine-Tuned  Improvement
----------------------------------------------------------
rouge1             0.4657       0.7523        61.6%
rouge2             0.3560       0.6106        71.5%
rougeL             0.4242       0.7168        69.0%


In [21]:
# Save evaluation results to JSON
#
# WHY: These numbers go in your:
#   - GitHub README ("ROUGE-L improved from X to Y")
#   - HF Model Card
#   - Medium blog post
#   - LinkedIn post
# Save them now so you don't have to re-run evaluation later.

import json

results = {
    "model": "Emar7/phi4-finance-finetuned",
    "base_model": model_id,
    "dataset": "virattt/financial-qa-10K",
    "eval_samples": 100,
    "scores": {
        "base":      {m: float(np.mean(base_scores[m])) for m in ["rouge1","rouge2","rougeL"]},
        "finetuned": {m: float(np.mean(ft_scores[m]))   for m in ["rouge1","rouge2","rougeL"]}
    }
}

with open("./eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Evaluation results saved to eval_results.json")
print(json.dumps(results, indent=2))

Evaluation results saved to eval_results.json
{
  "model": "Emar7/phi4-finance-finetuned",
  "base_model": "microsoft/Phi-4-mini-instruct",
  "dataset": "virattt/financial-qa-10K",
  "eval_samples": 100,
  "scores": {
    "base": {
      "rouge1": 0.4656655874270662,
      "rouge2": 0.35603411932661444,
      "rougeL": 0.4242266781407594
    },
    "finetuned": {
      "rouge1": 0.7522989779940632,
      "rouge2": 0.6106035248416039,
      "rougeL": 0.71675725166672
    }
  }
}


## PHASE 7 — INFERENCE DEMO

These outputs will be used directly in your dashboard, Medium blog, and LinkedIn post.

In [22]:
# Interactive finance demo function
#
# Use this to try any question you want after training.
# The output shows both models side by side for easy comparison.
# Great for screenshots to put in your blog post!

def finance_demo(question, context=""):
    print("\n" + "="*70)
    print(f"QUESTION:\n{question}")
    if context:
        print(f"\nCONTEXT (first 200 chars):\n{context[:200]}...")
    print("-"*70)

    print("BASE MODEL (no fine-tuning):")
    base_ans = generate_answer(base_only, tokenizer, question, context)
    print(base_ans)
    print()

    print("FINE-TUNED MODEL (trained on 10-K financial data):")
    ft_ans = generate_answer(eval_ft, tokenizer, question, context)
    print(ft_ans)
    print("="*70)

    return {"question": question, "base": base_ans, "finetuned": ft_ans}

# Run demo questions
finance_demo("What is the difference between gross profit and operating profit?")
finance_demo("How should an investor interpret declining operating margins?")
finance_demo("What does working capital tell us about a company's liquidity?")


QUESTION:
What is the difference between gross profit and operating profit?
----------------------------------------------------------------------
BASE MODEL (no fine-tuning):
Gross profit and operating profit are two different financial metrics used to evaluate a company's financial performance. Gross profit is the difference between revenue and the cost of goods sold (COGS). It measures the profitability of a company's core business activities, excluding overhead costs and operating expenses. Gross profit is calculated as follows: Gross Profit = Revenue - COGS.

Operating profit, on the other hand, is the difference between revenue and operating expenses, including COGS and operating expenses. It measures the profitability of a company's core business activities, excluding non-operating expenses such as interest and taxes. Operating profit is calculated as follows: Operating Profit = Revenue - Operating Expenses.

In summary, gross profit measures the profitability of a company's co

{'question': "What does working capital tell us about a company's liquidity?",
 'base': "Working capital is a financial metric that measures a company's short-term liquidity and operational efficiency. It is calculated as current assets minus current liabilities. A positive working capital indicates that a company has more current assets than current liabilities, suggesting it has sufficient liquidity to cover its short-term obligations and potentially invest in growth opportunities. Conversely, a negative working capital may signal potential liquidity issues, as the company might struggle to meet its short-term liabilities with its available assets. Therefore, working capital is a crucial indicator of a company's financial health and its ability to manage day-to-day operations effectively.",
 'finetuned': "Working capital indicates the company's ability to meet short-term obligations with its current assets."}

In [23]:
# Save demo outputs to JSON
#
# These outputs will be displayed in your interactive dashboard
# and quoted in your Medium blog post.

demo_questions = [
    "What is the difference between gross profit and operating profit?",
    "How should an investor interpret declining operating margins?",
    "What does working capital tell us about a company's liquidity?",
    "What are the key risks disclosed in a typical 10-K filing?",
    "How do you evaluate if a company is undervalued using financial ratios?"
]

demo_results = []
for q in demo_questions:
    result = finance_demo(q)
    demo_results.append(result)

with open("./demo_outputs.json", "w") as f:
    json.dump(demo_results, f, indent=2)

print("\nDemo outputs saved to demo_outputs.json")


QUESTION:
What is the difference between gross profit and operating profit?
----------------------------------------------------------------------
BASE MODEL (no fine-tuning):
Gross profit and operating profit are two different financial metrics used to assess a company's financial performance. Gross profit is calculated by subtracting the cost of goods sold (COGS) from total revenue. It measures the profitability of a company's core business activities, excluding indirect costs like administrative expenses, marketing costs, and interest expenses. Gross profit is often used to evaluate a company's pricing strategy and cost control measures.

Operating profit, on the other hand, is calculated by subtracting operating expenses (including COGS, administrative expenses, marketing costs, and interest expenses) from total revenue. It measures the profitability of a company's core business activities, excluding non-operating expenses like interest expenses and taxes. Operating profit is ofte